# Multiple arms kinematics
- Generate n_world x horizon x DOF array (= Trajectories)
- Put joint configuration to MjWarp data & forward
- View

In [1]:
# 1-1. import libraries
import mujoco
import mujoco.viewer
import mujoco_warp as mjw

import os
import sys
import time
import glfw
import numpy as np
import warp as wp

# 1-2. import pp base mujoco
sys.path.append(os.path.abspath('./'))
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.VIEWER import MUJOCOGLVIEWER

In [2]:
# 1-3. get model & data
xml_path = './asset/panda_scene.xml'
xml_abs_path = os.path.abspath(xml_path)
model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

In [3]:
joint_names = get_joint_names(model, data)
print("joint_names:", joint_names)
initial_qpos = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0])
apply_qpos_names(model, data, joint_names, initial_qpos)
mujoco.mj_forward(model, data)

joint_names: ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6', 'joint7']


In [4]:
def apply_qpos_idxs (model, data, idxs, value):
    if len(idxs) != len(value):
        raise ValueError("length of name and value is different")
    qpos_ = np.zeros(model.nq) # number of qpos
    for i, idx in enumerate(idxs):
        qpos_[idx] = value[i]
    data.qpos = qpos_    
    return None

def apply_qpos_names (model, data, names, value):
    if len(names) != len(value):
        raise ValueError("length of names and value is different")
    # initialize
    indexs = [model.joint(joint_name).qposadr[0] for joint_name in names]
    qpos_ = np.zeros(model.nq) # number of qpos

    for i, idx in enumerate(indexs):
        qpos_[idx] = value[i]
    data.qpos = qpos_
    return None

### Warp kernels -gemini

In [ ]:
import warp as wp
import numpy as np

# --- 1. THE GPU KERNEL ---
@wp.kernel
def apply_qpos_kernel(
    qpos: wp.array(dtype=float, ndim=2),    # Shape: (n_world, model.nq)
    idxs: wp.array(dtype=int, ndim=1),      # Shape: (num_targets,)
    values: wp.array(dtype=float, ndim=2),  # Shape: (n_world, num_targets)
    nq: int
):
    # wp.tid() gives us the current thread ID, which corresponds to the world index
    world_idx = wp.tid()

    # 1. Zero out the entire qpos for this specific world
    for i in range(nq):
        qpos[world_idx, i] = 0.0

    # 2. Assign the new values to the target indices
    num_targets = idxs.shape[0]
    for i in range(num_targets):
        target_idx = idxs[i]
        qpos[world_idx, target_idx] = values[world_idx, i]


# --- 2. THE CPU LAUNCHER ---
def apply_batched_qpos(model, mjw_data, target_identifiers, values_np, is_names=False):
    """
    Applies qpos values to n parallel worlds.
    
    Args:
        model: Standard MuJoCo model (mjModel)
        mjw_data: MuJoCo Warp data struct containing the batched qpos
        target_identifiers: list of ints (idxs) OR list of strings (names)
        values_np: Numpy array of shape (n_world, len(target_identifiers))
        is_names: Boolean, set to True if target_identifiers are strings
    """
    n_world = values_np.shape[0]
    num_targets = len(target_identifiers)
    
    if values_np.shape[1] != num_targets:
        raise ValueError("Values array must have shape (n_world, len(target_identifiers))")

    # 1. Translate names to indices on the CPU if necessary
    if is_names:
        idxs = [model.joint(name).qposadr[0] for name in target_identifiers]
    else:
        idxs = target_identifiers

    # 2. Transfer indices and values to the GPU
    # Note: If idxs never change, you should allocate wp_idxs ONCE outside this 
    # function to save PCIe bandwidth!
    wp_idxs = wp.array(idxs, dtype=int)
    wp_values = wp.array(values_np, dtype=float)

    # 3. Launch the kernel
    # We launch 1 thread per world.
    wp.launch(
        kernel=apply_qpos_kernel,
        dim=n_world,
        inputs=[
            mjw_data.qpos,  # The batched qpos array from mjwarp
            wp_idxs,
            wp_values,
            model.nq
        ]
    )

In [ ]:
n_worlds = 1024

# Create your batched values: Shape (1024 worlds, 2 joints)
# e.g., setting joint A to 1.5 and joint B to -0.5 for all worlds
my_values = np.zeros((n_worlds, 2))
my_values[:, 0] = 1.5  
my_values[:, 1] = -0.5 

# Using Names
apply_batched_qpos(
    model, 
    mjw_data, 
    target_identifiers=["shoulder_pan_joint", "elbow_flex_joint"], 
    values_np=my_values, 
    is_names=True
)

# Using Indices
apply_batched_qpos(
    model, 
    mjw_data, 
    target_identifiers=[0, 3], 
    values_np=my_values, 
    is_names=False
)

### MjWarp

In [5]:
world_column = 4
world_row = 2
n_world = world_column * world_row
max_contact_per_world = 50
mjw_model = mjw.put_model(model)
mjw_data = mjw.put_data(model, data, nworld=n_world, nconmax=max_contact_per_world)

Warp 1.12.0.dev20260126 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3060" (12 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/juju/.cache/warp/1.12.0.dev20260126


In [6]:
# from render_util import get_rgb
from mujoco_warp._src.types import RenderContext

@wp.kernel
def unpack_rgb_kernel(
  # In:
  packed: wp.array2d(dtype=wp.uint32),
  rgb_adr: wp.array(dtype=int),
  camera_index: int,
  # Out:
  rgb_out: wp.array3d(dtype=wp.vec3),
):
  """Unpack ABGR uint32 packed pixel data into separate R, G, and B channels."""
  worldid, pixelid = wp.tid()

  xid = pixelid % rgb_out.shape[2]
  yid = pixelid // rgb_out.shape[2]

  rgb_adr_offset = rgb_adr[camera_index]
  val = packed[worldid, rgb_adr_offset + pixelid]
  b = wp.float32(val & wp.uint32(0xFF)) * wp.static(1.0 / 255.0)
  g = wp.float32((val >> wp.uint32(8)) & wp.uint32(0xFF)) * wp.static(1.0 / 255.0)
  r = wp.float32((val >> wp.uint32(16)) & wp.uint32(0xFF)) * wp.static(1.0 / 255.0)
  rgb_out[worldid, yid, xid] = wp.vec3(r, g, b)

def get_rgb(rc: RenderContext, camera_index: int, rgb_out: wp.array3d(dtype=wp.vec3)):
  """Get the RGB data output from the render context buffers for a given camera index.

  Args:
    rc: The render context on device.
    camera_index: The index of the camera to get the RGB data for.
    rgb_out: The output array to store the RGB data in, with shape (nworld, height, width).
  """
  wp.launch(
    unpack_rgb_kernel,
    dim=(rgb_out.shape[0], rgb_out.shape[1] * rgb_out.shape[2]),
    inputs=[rc.rgb_data, rc.rgb_adr, camera_index],
    outputs=[rgb_out],
  )


camera_index = 1
camera_resolution = (500, 500)
rgb_data = wp.zeros((n_world, camera_resolution[1], camera_resolution[0]), dtype=wp.vec3)

def mjwarp_render_rgb(mjw_model, mjw_data, rc, camera_index, camera_resolution, world_grid = (4, 2)):
    world_row, world_column = world_grid
    # Populate data fields for the current state
    mjw.refit_bvh(mjw_model, mjw_data, rc)
    mjw.render(mjw_model, mjw_data, rc)
    get_rgb(rc, camera_index=camera_index, rgb_out=rgb_data)
    # check 
    rgb_grid = rgb_data.numpy()
    print("RGB data shape:", rgb_grid.shape)
    mjw_nworld, col, row, rgb = rgb_grid.shape
    if mjw_nworld * col * row * rgb != world_row * world_column * camera_resolution[0] * camera_resolution[1]*3:
        raise ValueError(f"Unexpected RGB data shape: {rgb_grid.shape}, expected total size: {world_row * world_column * camera_resolution[0] * camera_resolution[1]}")
    else:
        rgb_grid = rgb_data.numpy().reshape(world_row, world_column, camera_resolution[1], camera_resolution[0], 3) # 3
        rgb_grid = rgb_grid.transpose(0, 2, 1, 3, 4)
        rgb_grid = rgb_grid.reshape(world_row * camera_resolution[0], world_column * camera_resolution[1], 3)
        return rgb_grid

In [7]:
from OpenGL.GL import *
import numpy as np
import glfw
import sys

start_rc = time.time()
rc = mjw.create_render_context(
 model,
 nworld=n_world,
 cam_res=camera_resolution,
 render_rgb=True,
 render_depth=True,
)
rc_creation_duration = time.time() - start_rc
print(f"Render context creation duration: {rc_creation_duration:.4f} seconds")


# --- 1. INITIALIZATION & SETUP ---
if not glfw.init():
    print("Failed to initialize GLFW")
    sys.exit()

alpha = 0.7
gl_height = int(world_row * camera_resolution[1])
gl_width = int(world_column * camera_resolution[0])

window = glfw.create_window(gl_width, gl_height, "MuJoCo Warp Live", None, None)
if not window:
    glfw.terminate()
    print("Failed to create GLFW window")
    sys.exit()

glfw.make_context_current(window)

# Create and bind the texture ID
texture = glGenTextures(1)
glBindTexture(GL_TEXTURE_2D, texture)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glPixelStorei(GL_UNPACK_ALIGNMENT, 4)

# ALLOCATE GPU MEMORY ONCE (Passing 'None' creates an empty texture of the right size)
glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, gl_width, gl_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, None)

# Set up blending and background color once
glClearColor(0.2, 0.2, 0.2, 1.0) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_TEXTURE_2D)

# Pre-calculate alpha channel to save CPU time inside the loop
alpha_value = int(alpha * 255) 
alpha_channel = np.full((gl_height, gl_width, 1), alpha_value, dtype=np.uint8)

print("Starting render loop. Close the window to exit.")

# --- 2. THE MAIN LOOP ---
try:
    while not glfw.window_should_close(window):

        # get glfw window size & update camera resolution 
        gl_width_curr, gl_height_curr = glfw.get_framebuffer_size(window)
        # if gl width and height changed
        if gl_width_curr != gl_width or gl_height_curr != gl_height:
            gl_width, gl_height = gl_width_curr, gl_height_curr
            camera_resolution = (int(gl_width_curr/world_column), int(gl_height_curr/world_row))
            # update render context 
            rc = mjw.create_render_context(
                model,
                nworld=n_world,
                cam_res=camera_resolution,
                render_rgb=True,
                render_depth=True,
                )

        # [!] UPDATE YOUR DATA HERE [!]
        start_time = time.time()
        tick = 10
        for _ in range(tick):
            mjw.step(mjw_model, mjw_data)
        duration = time.time() - start_time
        print("==================================================================")
        print(f"mjw.step duration: {duration:.4f} seconds")
        rgb_grid = mjwarp_render_rgb(mjw_model, mjw_data, rc, camera_index, camera_resolution, world_grid=(world_row, world_column))
        render_duration = time.time() - start_time
        print(f"Render and RGB fetch duration: {render_duration:.4f} seconds")
        # this function is so slow
        
        # Format the new frame
        pixels_uint8_rgb = (rgb_grid * 255).astype(np.uint8)
        uint_change_duration = time.time() - start_time
        print(f"RGB uint8 conversion duration: {uint_change_duration:.4f} seconds")
        pixels_rgba = np.concatenate((pixels_uint8_rgb, alpha_channel), axis=2)
        alpha_concat_duration = time.time() - start_time
        print(f"Alpha channel concatenation duration: {alpha_concat_duration:.4f} seconds")
        pixels_rgba = np.ascontiguousarray(pixels_rgba, dtype=np.uint8)
        pixel_calculation_duration = time.time() - start_time
        print(f"Pixel formatting duration: {pixel_calculation_duration:.4f} seconds") # this takes really long time
        
        # UPDATE TEXTURE DATA (Fast: overwrites existing memory)
        glBindTexture(GL_TEXTURE_2D, texture)
        glTexSubImage2D(GL_TEXTURE_2D, 0, 0, 0, gl_width, gl_height, GL_RGBA, GL_UNSIGNED_BYTE, pixels_rgba)
        gl_texture_duration = time.time() - start_time

        # Clear screen
        glClear(GL_COLOR_BUFFER_BIT)

        # Draw the Quad
        glBegin(GL_QUADS)
        glTexCoord2f(0.0, 1.0); glVertex2f(-1.0, -1.0)
        glTexCoord2f(1.0, 1.0); glVertex2f( 1.0, -1.0)
        glTexCoord2f(1.0, 0.0); glVertex2f( 1.0,  1.0)
        glTexCoord2f(0.0, 0.0); glVertex2f(-1.0,  1.0)
        glEnd()
        gl_draw_duration = time.time() - start_time

        # Swap front and back buffers
        glfw.swap_buffers(window)
        
        # Poll for and process events (like window closing, mouse clicks, etc.)
        glfw.poll_events()

        total_duration = time.time() - start_time
        print(f"Total loop duration: {total_duration:.4f} seconds")

except KeyboardInterrupt:
    # Allows you to safely stop the loop in Jupyter by pressing "Interrupt Kernel"
    print("Render loop interrupted by user.")

# --- 3. CLEANUP ---
# Always properly destroy the window when done, especially in Jupyter!
glfw.destroy_window(window)
glfw.terminate()
print("GLFW terminated safely.")

Module mujoco_warp._src.render_util 815a8d5 load on device 'cuda:0' took 0.28 ms  (cached)
Module mujoco_warp._src.io db47b00 load on device 'cuda:0' took 0.46 ms  (cached)
Module mujoco_warp._src.bvh 3505354 load on device 'cuda:0' took 0.27 ms  (cached)
Render context creation duration: 0.1282 seconds
Starting render loop. Close the window to exit.
Module mujoco_warp._src.smooth 64ad9e5 load on device 'cuda:0' took 4.84 ms  (cached)
Module mujoco_warp._src.collision_driver de6dc98 load on device 'cuda:0' took 0.27 ms  (cached)
Module _nxn_broadphase__locals__kernel_36c68517 36c6851 load on device 'cuda:0' took 0.26 ms  (cached)
Module ccd_kernel_builder__locals__ccd_kernel_275b2300 275b230 load on device 'cuda:0' took 0.80 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_30157060 b2e79f4 load on device 'cuda:0' took 0.84 ms  (cached)
Module mujoco_warp._src.constraint 1ddad5e load on device 'cuda:0' took 0.43 ms  (cached)
Module _actuator_velocity__locals__ac